# Final Model and Cluster Profiling

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Load and Prepare the Final Data

In [ ]:
DATA_PATH = "../data/Dataset_Eating_Disorder.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df.rename(columns={
    "DesireToBuy _FromSnackBarOrCafe": "DesireToBuy_FromSnackBarOrCafe"
})
df_unique = df.drop_duplicates().copy()

print("Original shape:", df.shape)
print("Unique shape:", df_unique.shape)

In [ ]:
behavioral_groups = {
    "Restrained Eating": [
        "EatLess_OnWeightGain", "EatLess_AtMealtime", "RefuseFood_WeightConcern", "Monitor_Food",
        "Eat_SlimmingFoods", "EatLess_AfterOvereating", "EatLess_ToPreventWeightGain",
        "AvoidSnacks_BetweenMealsToWatchWeight", "AvoidEveningEating_ToWatchWeight",
        "ConsiderWeight_WhenEating"
    ],
    "Emotional Eating": [
        "Eat_WhenIrritated", "Eat_WhenUnoccupied", "Eat_WhenDepressedOrDiscouraged", "Eat_WhenLonely",
        "Eat_WhenSomeoneLetDown", "Eat_WhenAngry", "Eat_WhenExpectingBad", "Eat_WhenAnxious",
        "Eat_WhenThingsGoWrong", "Eat_WhenFrightened", "Eat_WhenDisappointed",
        "Eat_WhenEmotionallyUpset", "Eat_WhenBoredOrRestless"
    ],
    "External / Food-cue Eating": [
        "EatMore_IfFoodTasty", "EatMore_IfFoodSmellsOrLooksGood", "Eat_WhenSeeDeliciousFood",
        "Eat_DeliciousFoodImmediately", "DesireToBuy_FromBakery", "DesireToBuy_FromSnackBarOrCafe",
        "DesireToEat_WhenSeeOthersEating", "Resist_DeliciousFood", "EatMore_WhenSeeOthersEating",
        "Eat_WhenPreparingMeal"
    ],
    "Body Image / Shape Concern": [
        "Days_DesireForFlatStomach", "Days_FeltFat", "Days_WeightAffectedSelfJudgment",
        "Days_ShapeAffectedSelfJudgment", "Days_DissatisfiedWithWeight", "Days_DissatisfiedWithShape",
        "Days_UncomfortableToSeeOwnBody", "Days_UncomfortableBecauseOthersSeeingShape",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FastedControlShapeOrWeight",
        "Days_ExcludedFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FearLosingControlOverEating"
    ],
    "Habitual Eating": [
        "Eat_SpecificFoodsHabitually", "Location_TriggersHabitualEating",
        "AutomaticEating_WhenExperiencingStrongEmotion", "Realize_AfterEatingOutOfHabit"
    ]
}

behavioral_features = [question for questions in behavioral_groups.values() for question in questions]

final_questions = [
    "Days_FearLosingControlOverEating",
    "Days_ExcludedFoodControlShapeOrWeight",
    "Days_TriedLimitFoodControlShapeOrWeight",
    "Days_FollowedRulesControlShapeOrWeight",
    "Days_FeltFat",
    "EatLess_ToPreventWeightGain",
    "EatLess_AfterOvereating",
    "AvoidEveningEating_ToWatchWeight",
    "Eat_WhenAnxious",
    "Eat_WhenThingsGoWrong"
]

final_domains = {
    "Body Image / Shape Concern": final_questions[:5],
    "Restrained Eating": final_questions[5:8],
    "Emotional Eating": final_questions[8:10]
}

assert len(behavioral_features) == 50
assert len(final_questions) == 10

In [ ]:
scale_a_features = [question for question in behavioral_features if not question.startswith("Days_")]
scale_b_features = [question for question in behavioral_features if question.startswith("Days_")]

scale_a_mapping = {
    "Never": 0, "Seldom": 1, "Sometimes": 2, "Often": 3, "Very often": 4
}
scale_b_mapping = {
    "No days": 0, "1-5 days": 1, "6-12 days": 2, "13-15 days": 3, "Every day": 4
}

df_processed = df_unique.copy()

for column in scale_a_features:
    df_processed[column] = df_processed[column].map(scale_a_mapping)

for column in scale_b_features:
    df_processed[column] = df_processed[column].map(scale_b_mapping)

behavioral_data = df_processed[behavioral_features].copy()
X_final = behavioral_data[final_questions].copy()

assert X_final.shape == (601, 10)
assert X_final.isna().sum().sum() == 0
assert X_final.min().min() == 0 and X_final.max().max() == 4

print("Final matrix:", X_final.shape)
print("Missing values:", X_final.isna().sum().sum())
print("Encoded range:", X_final.min().min(), "to", X_final.max().max())

## Train the Final Model

In [ ]:
final_model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

raw_labels = final_model.fit_predict(X_final)

## Basic Final Model Results

In [ ]:
raw_cluster_sizes = np.bincount(raw_labels)
final_silhouette = silhouette_score(X_final, raw_labels)
cluster_centers = pd.DataFrame(
    final_model.cluster_centers_,
    columns=final_questions,
    index=["raw_cluster_0", "raw_cluster_1"]
)

print("Matrix shape:", X_final.shape)
print("Raw cluster sizes:", raw_cluster_sizes)
print("Silhouette Score:", round(final_silhouette, 4))
cluster_centers.round(3)

## Profile the Final Questions

In [ ]:
question_profile = X_final.assign(raw_cluster=raw_labels).groupby("raw_cluster").mean().T
question_profile.columns = ["raw_cluster_0", "raw_cluster_1"]
question_profile.round(3)

## Behavioral Domain Profiling

In [ ]:
domain_scores = pd.DataFrame({
    domain: X_final[questions].mean(axis=1)
    for domain, questions in final_domains.items()
})

raw_domain_profile = domain_scores.assign(raw_cluster=raw_labels).groupby("raw_cluster").mean().T
raw_domain_profile.columns = ["raw_cluster_0", "raw_cluster_1"]
raw_domain_profile.round(3)

## Semantic Cluster Naming

In [ ]:
cluster_center_means = final_model.cluster_centers_.mean(axis=1)
higher_cluster = int(cluster_center_means.argmax())
lower_cluster = int(cluster_center_means.argmin())

semantic_mapping = {
    higher_cluster: "higher_concern",
    lower_cluster: "lower_concern"
}
semantic_labels = pd.Series(raw_labels).map(semantic_mapping).to_numpy()

print("Raw cluster center means:", cluster_center_means.round(3))
print("Semantic mapping:", semantic_mapping)
print("Semantic sizes:", pd.Series(semantic_labels).value_counts().to_dict())

## Semantic Profile Tables

In [ ]:
semantic_columns = {
    f"raw_cluster_{raw_id}": semantic_label
    for raw_id, semantic_label in semantic_mapping.items()
}

semantic_question_profile = question_profile.rename(columns=semantic_columns)
semantic_domain_profile = raw_domain_profile.rename(columns=semantic_columns)

semantic_question_profile = semantic_question_profile[["higher_concern", "lower_concern"]]
semantic_domain_profile = semantic_domain_profile[["higher_concern", "lower_concern"]]

print("Semantic cluster sizes:")
print(pd.Series(semantic_labels).value_counts())
semantic_domain_profile.round(3)

In [ ]:
semantic_question_profile.plot(kind="barh", figsize=(8, 6))
plt.xlabel("Mean encoded response (0-4)")
plt.ylabel("")
plt.title("Final-question Profile by Semantic Cluster")
plt.xlim(0, 4)
plt.tight_layout()
plt.show()

In [ ]:
semantic_domain_profile.plot(kind="bar", figsize=(8, 4))
plt.ylabel("Mean encoded response (0-4)")
plt.xlabel("")
plt.title("Behavioral-domain Profile by Semantic Cluster")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 4)
plt.tight_layout()
plt.show()

## External Behavioral Validation

In [ ]:
remaining_questions = [question for question in behavioral_features if question not in final_questions]
external_rows = []

for question in remaining_questions:
    group_0 = behavioral_data.loc[raw_labels == 0, question]
    group_1 = behavioral_data.loc[raw_labels == 1, question]
    u_statistic = mannwhitneyu(group_0, group_1).statistic
    effect_size = 1 - 2 * u_statistic / (len(group_0) * len(group_1))

    external_rows.append({
        "Question Code": question,
        "Mean Difference (higher - lower)": (
            behavioral_data.loc[raw_labels == higher_cluster, question].mean()
            - behavioral_data.loc[raw_labels == lower_cluster, question].mean()
        ),
        "Absolute Rank-biserial Effect": abs(effect_size)
    })

external_validation = pd.DataFrame(external_rows).sort_values(
    "Absolute Rank-biserial Effect", ascending=False
)

print("Remaining questions:", len(remaining_questions))
print("Questions with |effect| >= 0.50:", (external_validation["Absolute Rank-biserial Effect"] >= 0.50).sum())
external_validation.head(10).round(3)

## Demographic Description

In [ ]:
demographic_features = [
    "Age_Range", "Gender", "Education_Level", "Employment_Status", "Marital_Status"
]

demographic_rows = []
for feature in demographic_features:
    for semantic_cluster in ["higher_concern", "lower_concern"]:
        values = df_unique.loc[semantic_labels == semantic_cluster, feature]
        demographic_rows.append({
            "Variable": feature,
            "Semantic Cluster": semantic_cluster,
            "Most Common Category": values.mode().iloc[0],
            "Percentage": values.value_counts(normalize=True).iloc[0] * 100
        })

demographic_summary = pd.DataFrame(demographic_rows)
demographic_summary.round({"Percentage": 1})

## Self-Perception Check

In [ ]:
self_perception_table = pd.crosstab(
    df_unique["Perception_EatingDisorder"],
    semantic_labels,
    normalize="columns"
) * 100

self_perception_table = self_perception_table[["higher_concern", "lower_concern"]]
self_perception_table.round(1)

## Summary

- Final matrix: **601 observations × 10 questions**
- Algorithm: **K-Means**
- Number of clusters: **2**
- Configuration: `random_state=42`, `n_init=10`
- Semantic labels: `higher_concern` and `lower_concern`
- Main distinction: relatively higher versus lower response intensity across body-image/shape concern, restrained eating, and emotional eating